In [103]:
import pandas as pd

In [104]:
df = pd.read_csv('../data/Titanic-Dataset.csv')

In [105]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [106]:
df.drop(columns=['PassengerId', 'Cabin', 'Name', 'Ticket'], inplace=True)

In [107]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='str')

In [108]:
from sklearn.model_selection import train_test_split

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Plan
Age -> Simple Imputer (Median)

Embarked -> Simple Imputer (Mode)

Sex -> Map into 0/1

Embarked -> One hot encoder

In [109]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

### 1. Imputation

In [110]:
processor = ColumnTransformer(
    transformers=[
        (
            'age_imputer',
            SimpleImputer(strategy='median'),
            ['Age']
        ),
        (
            'embarked_imputer',
            SimpleImputer(strategy='most_frequent'),
            ['Embarked']
        )
    ],
    remainder='passthrough'
)

In [111]:
X_train_step1 = processor.fit_transform(X_train)
X_test_step1 = processor.transform(X_test)

In [112]:
X_train_processed_df = pd.DataFrame(
    X_train_step1,
    columns=processor.get_feature_names_out(),
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_step1,
    columns=processor.get_feature_names_out(),
    index=X_test.index
)

In [113]:
X_train_processed_df.head()

,age_imputer__Age,embarked_imputer__Embarked,remainder__Pclass,remainder__Sex,remainder__SibSp,remainder__Parch,remainder__Fare
692,28.5,S,3,male,0,0,56.4958
481,28.5,S,2,male,0,0,0.0
527,28.5,S,1,male,0,0,221.7792
855,18.0,S,3,female,0,1,9.35
801,31.0,S,2,female,1,1,26.25


### 2. Encoding

In [114]:
processor2 = ColumnTransformer(
    transformers=[
        ('ohe_encoding', OneHotEncoder(drop='first'), ['remainder__Sex', 'embarked_imputer__Embarked'])
    ],
    remainder='passthrough'
)

In [115]:
X_train_processed = processor2.fit_transform(X_train_processed_df)
X_test_processed = processor2.transform(X_test_processed_df)

In [116]:
X_train_processed_df.isna().sum()

age_imputer__Age              0
embarked_imputer__Embarked    0
remainder__Pclass             0
remainder__Sex                0
remainder__SibSp              0
remainder__Parch              0
remainder__Fare               0
dtype: int64

### 3. Testing on a basic model

In [117]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8044692737430168

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


Confusion Matrix:
 [[98 12]
 [23 46]]
